# Phase 18 — Unlabeled Target Self-Training for Cross-Dataset NIDS

**NetShield performance-improvement phase**

The within-dataset controls are now strong (UNSW→UNSW ≈ F1 0.893; CICIDS→CICIDS ≈ F1 0.960), while the best earlier UNSW→CICIDS transfer was only about F1 0.339. Earlier phases already tested semantic alignment, expanded semantic features, CORAL, class weighting, and source-validation threshold transfer.

## New methodology
1. Train the established UNSW source model.
2. Align source/target representations with CORAL using unlabeled CICIDS adaptation rows.
3. Generate only very high-confidence target pseudo-labels.
4. Retrain on real UNSW labels + down-weighted CICIDS pseudo-labels.
5. Keep the CICIDS evaluation set completely frozen.
6. Select thresholds using UNSW validation only.

**CICIDS evaluation labels are never used for fitting, pseudo-label generation, threshold selection, or model selection.**

In [ ]:
from pathlib import Path
import gc, json, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, confusion_matrix
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",200)
RANDOM_STATE=42
CICIDS_CHUNK_SIZE=100_000
TARGET_POOL_N=600_000
TARGET_EVAL_N=300_000
PSEUDO_POS_THRESHOLD=0.95
PSEUDO_NEG_THRESHOLD=0.05
PSEUDO_WEIGHT=0.25
RF_PARAMS=dict(n_estimators=250,random_state=RANDOM_STATE,n_jobs=-1,max_features="sqrt",min_samples_leaf=2)
def find_project_root():
    for base in [Path.cwd(),*Path.cwd().parents]:
        if (base/"data/raw/UNSW-NB15").exists() and (base/"data/raw/CICIDS2017").exists(): return base
    return Path.cwd()
PROJECT_ROOT=find_project_root()
UNSW_ROOT=PROJECT_ROOT/"data/raw/UNSW-NB15"
CICIDS_ROOT=PROJECT_ROOT/"data/raw/CICIDS2017"
OUT=PROJECT_ROOT/"outputs/phase18_unlabeled_target_adaptation"
TABLES=OUT/"tables"
TABLES.mkdir(parents=True,exist_ok=True)
print("Project root:",PROJECT_ROOT)

In [ ]:
def norm(c): return str(c).replace("\ufeff","").strip()
def actual_col(columns,desired):
    d=desired.strip().lower()
    for c in columns:
        if norm(c).lower()==d: return c
    return None
train_files=list(UNSW_ROOT.rglob("UNSW_NB15_training-set.csv")); test_files=list(UNSW_ROOT.rglob("UNSW_NB15_testing-set.csv"))
if not train_files or not test_files: raise FileNotFoundError("UNSW official train/test CSVs not found")
UNSW_TRAIN_PATH=train_files[0]; UNSW_TEST_PATH=test_files[0]
cicids_files=sorted(CICIDS_ROOT.rglob("*.csv"))
if not cicids_files: raise FileNotFoundError("No CICIDS CSV files found")
print("UNSW:",UNSW_TRAIN_PATH,UNSW_TEST_PATH); print("CICIDS files:",len(cicids_files))

In [ ]:
unsw_train=pd.read_csv(UNSW_TRAIN_PATH,low_memory=False); unsw_test=pd.read_csv(UNSW_TEST_PATH,low_memory=False)
unsw_train.columns=[norm(c) for c in unsw_train.columns]; unsw_test.columns=[norm(c) for c in unsw_test.columns]
lt=actual_col(unsw_train.columns,"label"); lv=actual_col(unsw_test.columns,"label")
unsw_train["label_binary"]=pd.to_numeric(unsw_train[lt],errors="coerce").fillna(0).astype("int8")
unsw_test["label_binary"]=pd.to_numeric(unsw_test[lv],errors="coerce").fillna(0).astype("int8")
print(unsw_train.shape,unsw_test.shape)

In [ ]:
COMMON_11=["flow_duration_sec","fwd_packets","bwd_packets","fwd_bytes","bwd_bytes","total_packets","total_bytes","packet_rate","fwd_packet_rate","bwd_packet_rate","avg_packet_size"]
def num(df,c): return pd.to_numeric(df[c],errors="coerce")
def safe_div(a,b):
    a=pd.to_numeric(a,errors="coerce"); b=pd.to_numeric(b,errors="coerce"); return a.div(b.mask(b==0,np.nan))
def build_unsw_common(df):
    x=pd.DataFrame(index=df.index); x["flow_duration_sec"]=num(df,"dur"); x["fwd_packets"]=num(df,"spkts"); x["bwd_packets"]=num(df,"dpkts"); x["fwd_bytes"]=num(df,"sbytes"); x["bwd_bytes"]=num(df,"dbytes"); x["total_packets"]=x.fwd_packets+x.bwd_packets; x["total_bytes"]=x.fwd_bytes+x.bwd_bytes; x["packet_rate"]=safe_div(x.total_packets,x.flow_duration_sec); x["fwd_packet_rate"]=safe_div(x.fwd_packets,x.flow_duration_sec); x["bwd_packet_rate"]=safe_div(x.bwd_packets,x.flow_duration_sec); x["avg_packet_size"]=safe_div(x.total_bytes,x.total_packets); return x.replace([np.inf,-np.inf],np.nan).astype("float32")
def build_cicids_common(df):
    cols={norm(c).lower():c for c in df.columns}; req=["flow duration","total fwd packets","total backward packets","total length of fwd packets","total length of bwd packets","flow packets/s","fwd packets/s","bwd packets/s","average packet size"]
    miss=[c for c in req if c not in cols]
    if miss: raise KeyError(f"Missing CICIDS semantic columns: {miss}")
    x=pd.DataFrame(index=df.index); x["flow_duration_sec"]=safe_div(num(df,cols["flow duration"]),1_000_000); x["fwd_packets"]=num(df,cols["total fwd packets"]); x["bwd_packets"]=num(df,cols["total backward packets"]); x["fwd_bytes"]=num(df,cols["total length of fwd packets"]); x["bwd_bytes"]=num(df,cols["total length of bwd packets"]); x["total_packets"]=x.fwd_packets+x.bwd_packets; x["total_bytes"]=x.fwd_bytes+x.bwd_bytes; x["packet_rate"]=num(df,cols["flow packets/s"]); x["fwd_packet_rate"]=num(df,cols["fwd packets/s"]); x["bwd_packet_rate"]=num(df,cols["bwd packets/s"]); x["avg_packet_size"]=num(df,cols["average packet size"]); return x.replace([np.inf,-np.inf],np.nan).astype("float32")
X_unsw=build_unsw_common(unsw_train); y_unsw=unsw_train.label_binary.to_numpy(dtype="int8")
print("UNSW semantic:",X_unsw.shape)

In [ ]:
file_info={}; file_rows={}
for path in cicids_files:
    hdr=pd.read_csv(path,nrows=0,low_memory=True); hdr.columns=[norm(c) for c in hdr.columns]; lab=actual_col(hdr.columns,"Label")
    if lab is None: raise KeyError(f"No Label column found in {path.name}")
    file_info[path]={"columns":hdr.columns.tolist(),"label":lab}
    n=0
    for ch in pd.read_csv(path,usecols=[lab],chunksize=CICIDS_CHUNK_SIZE,low_memory=True): n+=len(ch)
    file_rows[path]=n
total_rows=sum(file_rows.values()); pool_n=min(TARGET_POOL_N,total_rows)
quotas={}; rem=pool_n
for i,p in enumerate(cicids_files):
    q=rem if i==len(cicids_files)-1 else min(int(round(pool_n*file_rows[p]/total_rows)),rem); quotas[p]=q; rem-=q
parts=[]
for path in cicids_files:
    cols=file_info[path]["columns"]; label=file_info[path]["label"]
    def col(name):
        c=actual_col(cols,name)
        if c is None: raise KeyError(f"{name} missing from {path.name}")
        return c
    use=[col("Flow Duration"),col("Total Fwd Packets"),col("Total Backward Packets"),col("Total Length of Fwd Packets"),col("Total Length of Bwd Packets"),col("Flow Packets/s"),col("Fwd Packets/s"),col("Bwd Packets/s"),col("Average Packet Size"),label]
    chunks=[]
    for ch in pd.read_csv(path,usecols=use,chunksize=CICIDS_CHUNK_SIZE,low_memory=True):
        ch.columns=[norm(c) for c in ch.columns]; chunks.append(ch)
    df=pd.concat(chunks,ignore_index=True); df=df.sample(n=min(quotas[path],len(df)),random_state=RANDOM_STATE).copy(); df["label_binary"]=df[label].astype(str).str.strip().str.upper().ne("BENIGN").astype("int8"); parts.append(df); print("sampled",len(df),path.name); del chunks,df; gc.collect()
cicids_pool=pd.concat(parts,ignore_index=True); del parts; gc.collect()
if len(cicids_pool)>pool_n: cicids_pool=cicids_pool.sample(n=pool_n,random_state=RANDOM_STATE).reset_index(drop=True)
print("Target pool:",cicids_pool.shape,cicids_pool.label_binary.value_counts().to_dict())

In [ ]:
X_cic=build_cicids_common(cicids_pool); y_cic=cicids_pool.label_binary.to_numpy(dtype="int8")
adapt_idx,eval_idx=train_test_split(np.arange(len(X_cic)),test_size=min(300_000,len(X_cic)//2),random_state=RANDOM_STATE,stratify=y_cic)
X_adapt=X_cic.iloc[adapt_idx].reset_index(drop=True); X_eval=X_cic.iloc[eval_idx].reset_index(drop=True); y_eval=y_cic[eval_idx]
fit_idx,val_idx=train_test_split(np.arange(len(X_unsw)),test_size=.20,random_state=RANDOM_STATE,stratify=y_unsw)
X_fit=X_unsw.iloc[fit_idx]; X_val=X_unsw.iloc[val_idx]; y_fit=y_unsw[fit_idx]; y_val=y_unsw[val_idx]
imp=SimpleImputer(strategy="median"); X_fit_i=imp.fit_transform(X_fit).astype("float32"); X_val_i=imp.transform(X_val).astype("float32"); X_adapt_i=imp.transform(X_adapt).astype("float32"); X_eval_i=imp.transform(X_eval).astype("float32")
print("adapt",len(X_adapt),"eval",len(X_eval),"eval attack rate",round(float(y_eval.mean()),6))

In [ ]:
def sym_power(C,power,eps=1e-6):
    vals,vecs=np.linalg.eigh(C); vals=np.clip(vals,eps,None); return (vecs*(vals**power))@vecs.T
def fit_coral(Xs,Xt):
    ms,mt=Xs.mean(0),Xt.mean(0); Cs=np.cov(Xs-ms,rowvar=False)+np.eye(Xs.shape[1])*1e-5; Ct=np.cov(Xt-mt,rowvar=False)+np.eye(Xt.shape[1])*1e-5; A=sym_power(Cs,-.5)@sym_power(Ct,.5); return {"ms":ms,"mt":mt,"A":A}
def coral_source(X,c): return (X-c["ms"])@c["A"]+c["mt"]
coral=fit_coral(X_fit_i,X_adapt_i); X_fit_c=coral_source(X_fit_i,coral); X_val_c=coral_source(X_val_i,coral); X_adapt_c=X_adapt_i; X_eval_c=X_eval_i
C1=np.cov(X_fit_i-X_fit_i.mean(0),rowvar=False); C2=np.cov(X_adapt_i-X_adapt_i.mean(0),rowvar=False); C3=np.cov(X_fit_c-X_fit_c.mean(0),rowvar=False)
print("Cov distance before:",float(np.linalg.norm(C1-C2,ord="fro"))); print("Cov distance after :",float(np.linalg.norm(C3-C2,ord="fro")))

In [ ]:
def choose_threshold(y,p):
    ts=np.round(np.arange(.05,.951,.01),2); scores=[f1_score(y,(p>=t).astype("int8"),zero_division=0) for t in ts]; return float(ts[int(np.argmax(scores))])
def evaluate(y,p,t):
    pred=(p>=t).astype("int8"); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel(); return {"accuracy":accuracy_score(y,pred),"precision":precision_score(y,pred,zero_division=0),"recall":recall_score(y,pred,zero_division=0),"f1":f1_score(y,pred,zero_division=0),"macro_f1":f1_score(y,pred,average="macro",zero_division=0),"pr_auc":average_precision_score(y,p),"roc_auc":roc_auc_score(y,p),"fpr":fp/(fp+tn) if fp+tn else 0,"tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),"predicted_attack_rate":float(pred.mean()),"threshold":t}

In [ ]:
rows=[]
rf0=RandomForestClassifier(**RF_PARAMS,class_weight=None); t=time.perf_counter(); rf0.fit(X_fit_i,y_fit); s0=time.perf_counter()-t; p=rf0.predict_proba(X_val_i)[:,1]; th=choose_threshold(y_val,p); pe=rf0.predict_proba(X_eval_i)[:,1]; r=evaluate(y_eval,pe,th); r.update(method="baseline",pseudo_rows=0,train_seconds=s0); rows.append(r)
rf1=RandomForestClassifier(**RF_PARAMS,class_weight="balanced"); t=time.perf_counter(); rf1.fit(X_fit_c,y_fit); s1=time.perf_counter()-t; p=rf1.predict_proba(X_val_c)[:,1]; th=choose_threshold(y_val,p); pe=rf1.predict_proba(X_eval_c)[:,1]; r=evaluate(y_eval,pe,th); r.update(method="CORAL + balanced",pseudo_rows=0,train_seconds=s1); rows.append(r)
bench=pd.DataFrame(rows); display(bench[["method","f1","macro_f1","precision","recall","pr_auc","roc_auc","fpr","threshold","predicted_attack_rate"]])

## Main intervention — conservative unlabeled target self-training

Pseudo-labels use only the CORAL+balanced model's confidence on the unlabeled adaptation partition. Positive ≥ 0.95; negative ≤ 0.05; uncertain rows are discarded. Pseudo-labels receive weight 0.25.

In [ ]:
p_adapt=rf1.predict_proba(X_adapt_c)[:,1]; mask=(p_adapt>=PSEUDO_POS_THRESHOLD)|(p_adapt<=PSEUDO_NEG_THRESHOLD); X_pseudo=X_adapt_c[mask]; y_pseudo=(p_adapt[mask]>=PSEUDO_POS_THRESHOLD).astype("int8")
print("pseudo rows",len(y_pseudo),"coverage",round(len(y_pseudo)/len(X_adapt_c),6),"attack rate",round(float(y_pseudo.mean()),6) if len(y_pseudo) else np.nan)
if len(y_pseudo)<1000: raise ValueError("Too few confident pseudo-labels; self-training should not proceed.")
X_aug=np.vstack([X_fit_c,X_pseudo]); y_aug=np.concatenate([y_fit,y_pseudo]); w=np.concatenate([np.ones(len(y_fit),dtype="float32"),np.full(len(y_pseudo),PSEUDO_WEIGHT,dtype="float32")])
rf2=RandomForestClassifier(**RF_PARAMS,class_weight="balanced"); t=time.perf_counter(); rf2.fit(X_aug,y_aug,sample_weight=w); s2=time.perf_counter()-t; pv=rf2.predict_proba(X_val_c)[:,1]; th2=choose_threshold(y_val,pv); pe=rf2.predict_proba(X_eval_c)[:,1]; r2=evaluate(y_eval,pe,th2); r2.update(method="CORAL + balanced + self-training",pseudo_rows=len(y_pseudo),train_seconds=s2)
results=pd.concat([bench,pd.DataFrame([r2])],ignore_index=True); display(results[["method","f1","macro_f1","precision","recall","pr_auc","roc_auc","fpr","threshold","pseudo_rows"]].sort_values("f1",ascending=False))

## One fixed second iteration

A second pseudo-label refresh is allowed once, without tuning against target labels. This limits confirmation-bias amplification.

In [ ]:
p2=rf2.predict_proba(X_adapt_c)[:,1]; mask2=(p2>=PSEUDO_POS_THRESHOLD)|(p2<=PSEUDO_NEG_THRESHOLD); X_pseudo2=X_adapt_c[mask2]; y_pseudo2=(p2[mask2]>=PSEUDO_POS_THRESHOLD).astype("int8")
print("iteration-2 pseudo rows",len(y_pseudo2),"coverage",round(len(y_pseudo2)/len(X_adapt_c),6))
X_aug2=np.vstack([X_fit_c,X_pseudo2]); y_aug2=np.concatenate([y_fit,y_pseudo2]); w2=np.concatenate([np.ones(len(y_fit),dtype="float32"),np.full(len(y_pseudo2),PSEUDO_WEIGHT,dtype="float32")])
rf3=RandomForestClassifier(**RF_PARAMS,class_weight="balanced"); t=time.perf_counter(); rf3.fit(X_aug2,y_aug2,sample_weight=w2); s3=time.perf_counter()-t; pv=rf3.predict_proba(X_val_c)[:,1]; th3=choose_threshold(y_val,pv); pe=rf3.predict_proba(X_eval_c)[:,1]; r3=evaluate(y_eval,pe,th3); r3.update(method="CORAL + balanced + self-training x2",pseudo_rows=len(y_pseudo2),train_seconds=s3)
results=pd.concat([results,pd.DataFrame([r3])],ignore_index=True); display(results[["method","f1","macro_f1","precision","recall","pr_auc","roc_auc","fpr","threshold","pseudo_rows"]].sort_values("f1",ascending=False))

In [ ]:
benchmark_f1=float(results.loc[results.method=="CORAL + balanced","f1"].iloc[0]); results["f1_gain_vs_CORAL_balanced"]=results.f1-benchmark_f1
display(results[["method","f1","macro_f1","precision","recall","pr_auc","roc_auc","fpr","threshold","predicted_attack_rate","pseudo_rows","f1_gain_vs_CORAL_balanced"]].sort_values("f1",ascending=False))
best=results.sort_values(["f1","pr_auc","roc_auc"],ascending=False).iloc[0]; print("BEST:",best.method,"F1",round(best.f1,4),"gain",round(best.f1_gain_vs_CORAL_balanced,4),"PR-AUC",round(best.pr_auc,4),"FPR",round(best.fpr,4))

In [ ]:
results.to_csv(TABLES/"01_phase18_cross_dataset_results.csv",index=False)
pd.DataFrame([{ "iteration":1,"pseudo_rows":len(y_pseudo),"coverage":len(y_pseudo)/len(X_adapt_c),"pseudo_attack":int((y_pseudo==1).sum())},{"iteration":2,"pseudo_rows":len(y_pseudo2),"coverage":len(y_pseudo2)/len(X_adapt_c),"pseudo_attack":int((y_pseudo2==1).sum())}]).to_csv(TABLES/"02_pseudo_label_summary.csv",index=False)
config={"phase":18,"method":"unlabeled target-domain self-training after CORAL","random_state":RANDOM_STATE,"target_pool_n":int(len(cicids_pool)),"target_adaptation_n":int(len(X_adapt)),"target_evaluation_n":int(len(X_eval)),"features":COMMON_11,"rf_params":RF_PARAMS,"pseudo_positive_threshold":PSEUDO_POS_THRESHOLD,"pseudo_negative_threshold":PSEUDO_NEG_THRESHOLD,"pseudo_weight":PSEUDO_WEIGHT,"threshold_selection":"UNSW source validation only","target_labels":"CICIDS labels used only for final frozen evaluation"}
with open(TABLES/"03_phase18_config.json","w",encoding="utf-8") as f: json.dump(config,f,indent=2)
print("Saved outputs to",TABLES)

# Decision rule

A successful Phase 18 result must beat the established **CORAL + balanced** benchmark on F1 while keeping the precision/FPR trade-off defensible. If it does not, we stop rather than endlessly tune pseudo-label thresholds; the next intervention should use a different domain-adaptation mechanism.